In [1]:
import ee

ee.Authenticate()
ee.Initialize(project='avilight-483312')

# --- Time range ---
START_DATE = ee.Date('2014-01-01')
N_MONTHS = 144  # Jan 2014 through Dec 2025

# --- Google Drive export ---
DRIVE_FOLDER = 'Satellite Datasets'
EXPORT_NAME = 'province_alan_monthly_2014_2025'

# --- Philippine province boundaries ---
# GAUL 2025 replaces the deprecated 2015 boundary asset.
provinces = (
    ee.FeatureCollection('FAO/GAUL/2025/level2')
    .filter(ee.Filter.eq('GAUL0_NAME', 'Philippines'))
)

print('Province features:', provinces.size().getInfo())


def mask_alan(image):
    """
    Keep only high-quality Black Marble observations:
    Mandatory_Quality_Flag 0 or 1.
    """
    quality_mask = image.select('Mandatory_Quality_Flag').lte(1)

    return (
        image.select('DNB_BRDF_Corrected_NTL')
        .updateMask(quality_mask)
        .rename('alan')
    )


def monthly_alan(month_index):
    """Create one monthly-median ALAN image and valid-pixel band."""
    month_index = ee.Number(month_index)

    d0 = START_DATE.advance(month_index, 'month')
    d1 = d0.advance(1, 'month')

    monthly_collection = (
        ee.ImageCollection('NASA/VIIRS/002/VNP46A2')
        .filterDate(d0, d1)
        .map(mask_alan)
    )

    # Monthly median of quality-approved daily night-light observations.
    alan = monthly_collection.median().rename('alan')

    # 1 for valid monthly ALAN pixels; mean by province is valid-pixel share.
    alan_valid = alan.mask().rename('alan_valid')

    return (
        alan.addBands(alan_valid)
        .set({
            'year': d0.get('year'),
            'month': d0.get('month'),
            'date': d0.format('YYYY-MM')
        })
    )


# Build the 144 monthly images.
month_indices = ee.List.sequence(0, N_MONTHS - 1)

alan_monthly = ee.ImageCollection.fromImages(
    month_indices.map(monthly_alan)
)


def reduce_month_to_provinces(image):
    """Calculate province mean ALAN and valid-pixel share for one month."""
    reduced = image.reduceRegions(
        collection=provinces,
        reducer=ee.Reducer.mean(),
        scale=1000,
        tileScale=4
    )

    def format_row(feature):
        return ee.Feature(None, {
            'adm1': feature.get('GAUL1_NAME'),
            'province': feature.get('GAUL2_NAME'),
            'adm2_code': feature.get('GAUL2_CODE'),
            'year': image.get('year'),
            'month': image.get('month'),
            'alan': feature.get('alan'),
            'alan_valid': feature.get('alan_valid')
        })

    return reduced.map(format_row)


# One row per province × month.
province_alan_fc = alan_monthly.map(
    reduce_month_to_provinces
).flatten()

province_alan_export = province_alan_fc.select([
    'adm1',
    'province',
    'adm2_code',
    'year',
    'month',
    'alan',
    'alan_valid'
])

task = ee.batch.Export.table.toDrive(
    collection=province_alan_export,
    description=EXPORT_NAME,
    folder=DRIVE_FOLDER,
    fileNamePrefix=EXPORT_NAME,
    fileFormat='CSV'
)

task.start()

print('✅ Province-level monthly ALAN export started')
print('Task ID:', task.id)
print('Drive folder:', DRIVE_FOLDER)
print('Period: January 2014 to December 2025')

Province features: 85
✅ Province-level monthly ALAN export started
Task ID: AMAARLPHKTGGOP7ZW64QOEOK
Drive folder: Satellite Datasets
Period: January 2014 to December 2025


In [4]:
print(task.status())

{'state': 'COMPLETED', 'description': 'province_alan_monthly_2014_2025', 'priority': 100, 'creation_timestamp_ms': 1794716356245, 'update_timestamp_ms': 1794716521742, 'start_timestamp_ms': 1794716360050, 'task_type': 'EXPORT_FEATURES', 'destination_uris': ['https://drive.google.com/#folders/1-bU5d-UWd39u4OkyDb09gBu3DzCrayi9'], 'attempt': 1, 'batch_eecu_usage_seconds': 4535.9091796875, 'id': 'AMAARLPHKTGGOP7ZW64QOEOK', 'name': 'projects/avilight-483312/operations/AMAARLPHKTGGOP7ZW64QOEOK'}


In [5]:
import pandas as pd

df_alan = pd.read_csv('province_alan_monthly_2014_2025.csv')

print('Total rows:', len(df_alan))
print('Provinces:', df_alan['adm2_code'].nunique())
print('Months:', df_alan[['year', 'month']].drop_duplicates().shape[0])
print('Expected rows:', df_alan['adm2_code'].nunique() * 144)
print('Low-valid rows:', (df_alan['alan_valid'] < 0.3).sum())

Total rows: 12240
Provinces: 85
Months: 144
Expected rows: 12240
Low-valid rows: 10


In [6]:
df_alan.loc[df_alan['alan_valid'] < 0.3, 'alan'] = pd.NA
df_alan.to_parquet('province_alan_monthly.parquet', index=False)